# 🧬 Exploring NCBI with Biopython
This notebook demonstrates how to query NCBI databases using Biopython's `Entrez` module.

**Topics covered:**
1. Setup & configuration
2. Fetch a record by accession or GI
3. Parse GenBank features (exons, CDS, gene)
4. Generate a GTF-like exon annotation table
5. Retrieve the linked Gene ID
6. Fetch the protein product
7. Search NCBI databases with keywords


## 1. Setup
Install Biopython if needed, then set your email (required by NCBI).

In [ ]:
# !pip install biopython pandas
from Bio import Entrez, SeqIO
from Bio import SeqRecord
import pandas as pd
import sys
import re

# NCBI requires a valid email to use Entrez
Entrez.email = 'daniel.garciaruano@ibgc.cnrs.fr'

# Example identifiers from our sequence name: gi|637242346|ref|XM_008105585.1
#GI_NUMBER    = '637242346'
#ACCESSION    = 'XM_008105585.1'
ACCESSION = 'XM_013799913.2'  # Obsolete id
ACCESSION = 'XM_002873863.2'  # Not obsolete
ACCESSION = 'XM_001702112.1'  # Obsolete ID that does not work in snakemake pipeline

## 2. Fetch a GenBank Record
You can use either the GI number or the accession.version. 
NCBI recommends accession.version as GI numbers are legacy identifiers.

In [ ]:
# Fetch by accession (preferred)
handle = Entrez.efetch(db='nucleotide', id=ACCESSION, rettype='gb', retmode='text')
record = SeqIO.read(handle, 'genbank')
handle.close()

print(f'ID          : {record.id}')
print(f'Name        : {record.name}')
print(f'Description : {record.description}')
print(f'Sequence len: {len(record.seq)} bp')
print(f'Num features: {len(record.features)}')


ID          : XM_001702112.1
Name        : XM_001702112
Description : Chlamydomonas reinhardtii predicted protein (CHLREDRAFT_206223), mRNA
Sequence len: 920 bp
Num features: 3


## 3. Inspect All Feature Types
The FEATURES table in a GenBank record can contain genes, mRNA, exons, CDS, and more.

In [ ]:
from collections import Counter

feature_types = Counter(f.type for f in record.features)
print('Feature type counts:')
for ftype, count in feature_types.most_common():
    print(f'  {ftype:<15} {count}')


Feature type counts:
  source          1
  gene            1
  CDS             1


### 3.1 Print raw genbank

In [ ]:
SeqIO.write(record, sys.stdout, "genbank")

LOCUS       XM_001702112             920 bp    mRNA    linear   PLN 29-AUG-2017
DEFINITION  Chlamydomonas reinhardtii predicted protein (CHLREDRAFT_206223),
            mRNA.
ACCESSION   XM_001702112
VERSION     XM_001702112.1
DBLINK      BioProject: PRJNA21061
            BioSample: SAMN02953692
KEYWORDS    RefSeq.
SOURCE      Chlamydomonas reinhardtii
  ORGANISM  Chlamydomonas reinhardtii
            Eukaryota; Viridiplantae; Chlorophyta; core chlorophytes;
            Chlorophyceae; CS clade; Chlamydomonadales; Chlamydomonadaceae;
            Chlamydomonas.
REFERENCE   1  (bases 1 to 920)
  AUTHORS   Merchant,S.S., Prochnik,S.E., Vallon,O., Harris,E.H.,
            Karpowicz,S.J., Witman,G.B., Terry,A., Salamov,A.,
            Fritz-Laylin,L.K., Marechal-Drouard,L., Marshall,W.F., Qu,L.H.,
            Nelson,D.R., Sanderfoot,A.A., Spalding,M.H., Kapitonov,V.V., Ren,Q.,
            Ferris,P., Lindquist,E., Shapiro,H., Lucas,S.M., Grimwood,J.,
            Schmutz,J., Cardol,P., Cerutt

1

Extract genomic sequence accession (usually chromosome)

In [ ]:
def extract_genomic_accession(seq_record: SeqRecord) -> str:
    genomic_accession = None
    for xref in record.dbxrefs:
        prefixes = ['NC_', 'NT_', 'NW_', 'AC_']
        if xref.startswith(tuple(prefixes)):
            genomic_accession = xref
            break
    if not genomic_accession:
        # Look for genomic accession in the annotations
        if 'comment' in record.annotations:
            comment = record.annotations['comment']
            # Parse for genomic accession NC_, NT_, NW_ or AC_
            match = re.search(r'\b((?:NC|NT|NW|AC)_\d+\.\d+)\b', comment)
            if match:
                genomic_accession = match.group(1)
    return genomic_accession

genomic_accession = extract_genomic_accession(record)

In [ ]:
if genomic_accession:
    print(f"Genomic accession for id {ACCESSION} is {genomic_accession}")
else:
    raise ValueError(f"Unable to find genomic accession for transcript {ACCESSION}")

ValueError: Unable to find genomic accession for transcript XM_001702112.1

In [ ]:
# # Clean accession extraction from a SeqRecord (e.g., `genomic`)
# def extract_accession(seq_record: SeqRecord) -> str:
#     # Preferred source: SeqRecord.id
#     candidates = [
#         getattr(seq_record, "id", None),
#         getattr(seq_record, "name", None),
#         getattr(seq_record, "description", None),
#     ]
#     for c in candidates:
#         if c:
#             token = str(c).strip().split()[0]
#             if token and token != "<unknown":
#                 return token
#     raise ValueError("Could not extract accession from SeqRecord.")

# GENOMIC_ACCESSION = extract_accession(genomic)
# print(f"Extracted genomic accession: {GENOMIC_ACCESSION}")

In [ ]:
if not genomic_accession:
    genomic_accession = 'NW_001843905'

handle = Entrez.efetch(db="nucleotide", id=genomic_accession,
                    rettype="gb", retmode="text")
genomic = SeqIO.read(handle, "genbank")
handle.close()

# Filter exon/mRNA features belonging to this transcript
for feature in genomic.features:
    if feature.type in ("exon", "mRNA"):
        tx = feature.qualifiers.get("transcript_id", [""])[0]
        if ACCESSION in tx:
            print(f"{feature.type}: {feature.location}")

mRNA: join{[251815:252010](-), [251507:251589](-), [250691:251334](-)}


In [ ]:
genomic

SeqRecord(seq=Seq(None, length=404895), id='NW_001843905.1', name='NW_001843905', description='Chlamydomonas reinhardtii strain CC-503 cw92 mt+ CHLREscaffold_73 genomic scaffold, whole genome shotgun sequence', dbxrefs=['BioProject:PRJNA21061', 'BioSample:SAMN02953692', 'Assembly:GCF_000002595.1'])

In [ ]:
def get_assembly(genomic_record: SeqRecord) -> str:
    """
    Extract assembly accession from the dbxrefs field in genomic GenBank record
    """
    assembly_accession = None
    # Check dbxrefs in the genomic record
    for xref in genomic_record.dbxrefs:
        if xref.startswith('Assembly:'):
            assembly_accession = xref.split(':')[1]
            break
    return assembly_accession

assembly_accession = get_assembly(genomic)
print(f'Assembly accession from GenBank record: {assembly_accession}')

Assembly accession from GenBank record: GCF_000004255.2


In [ ]:
SeqIO.write(genomic, sys.stdout, "genbank")

LOCUS       NW_003302550        25113588 bp    DNA     linear   CON 11-MAY-2017
DEFINITION  Arabidopsis lyrata subsp. lyrata unplaced genomic scaffold
            ARALYscaffold_6, whole genome shotgun sequence.
ACCESSION   NW_003302550
VERSION     NW_003302550.1
DBLINK      BioProject: PRJNA49545
            BioSample: SAMN02981250
            Assembly: GCF_000004255.2
KEYWORDS    WGS; RefSeq.
SOURCE      Arabidopsis lyrata subsp. lyrata
  ORGANISM  Arabidopsis lyrata subsp. lyrata
            Eukaryota; Viridiplantae; Streptophyta; Embryophyta; Tracheophyta;
            Spermatophyta; Magnoliopsida; eudicotyledons; Gunneridae;
            Pentapetalae; rosids; malvids; Brassicales; Brassicaceae;
            Camelineae; Arabidopsis.
REFERENCE   1  (bases 1 to 25113588)
  AUTHORS   Hu,T.T., Pattyn,P., Bakker,E.G., Cao,J., Cheng,J.F., Clark,R.M.,
            Fahlgren,N., Fawcett,J.A., Grimwood,J., Gundlach,H., Haberer,G.,
            Hollister,J.D., Ossowski,S., Ottilar,R.P., Salamov,A.A

1

### 3.2 Get genome assembly

In [ ]:
from pathlib import Path
import urllib.request

ASSEMBLY = 'GCF_000004255.2'

# Resolve assembly FTP path via NCBI Entrez (Biopython)
handle = Entrez.esearch(db="assembly", term=f"{ASSEMBLY}[Assembly Accession]", retmax=1)
search = Entrez.read(handle)
handle.close()

if not search["IdList"]:
    raise ValueError(f"Assembly not found: {ASSEMBLY}")

assembly_uid = search["IdList"][0]

handle = Entrez.esummary(db="assembly", id=assembly_uid, report="full")
summary = Entrez.read(handle)
handle.close()

doc = summary["DocumentSummarySet"]["DocumentSummary"][0]
ftp_path = doc.get("FtpPath_RefSeq") or doc.get("FtpPath_GenBank")
if not ftp_path:
    raise ValueError(f"No FTP path available for assembly: {ASSEMBLY}")

prefix = ftp_path.rsplit("/", 1)[-1]
gtf_url = f"{ftp_path}/{prefix}_genomic.gtf.gz"
gtf_gz_path = f"{ASSEMBLY}_genomic.gtf.gz"

# Download only if missing
gtf_file = Path(gtf_gz_path)
if not gtf_file.exists():
    urllib.request.urlretrieve(gtf_url, gtf_gz_path)
    print(f"Downloaded: {gtf_gz_path}")
else:
    print(f"Already exists, skipping download: {gtf_gz_path}")

# Optional: load into a dataframe
if "gtf_cols" not in globals():
    gtf_cols = ["seqname", "source", "feature", "start", "end", "score", "strand", "frame", "attribute"]

df_gtf = pd.read_csv(
    gtf_gz_path,
    sep="\t",
    comment="#",
    names=gtf_cols,
    compression="gzip",
    low_memory=False
)

print(df_gtf.head())
print(f"Rows: {len(df_gtf):,}")

Downloaded: GCF_000004255.2_genomic.gtf.gz
          seqname  source     feature  start   end score strand frame  \
0  NW_003302555.1  Gnomon        gene      5  2879     .      -     .   
1  NW_003302555.1  Gnomon  transcript      5  2879     .      -     .   
2  NW_003302555.1  Gnomon        exon   2444  2879     .      -     .   
3  NW_003302555.1  Gnomon        exon   2124  2347     .      -     .   
4  NW_003302555.1  Gnomon        exon   1803  2035     .      -     .   

                                           attribute  
0  gene_id "LOC9330760"; transcript_id ""; db_xre...  
1  gene_id "LOC9330760"; transcript_id "XM_002891...  
2  gene_id "LOC9330760"; transcript_id "XM_002891...  
3  gene_id "LOC9330760"; transcript_id "XM_002891...  
4  gene_id "LOC9330760"; transcript_id "XM_002891...  
Rows: 650,095


In [ ]:
# Filter GTF for entries matching our transcript accession (exact transcript_id match)
tx_pattern = rf'transcript_id "{re.escape(ACCESSION)}"'
transcript_entries = df_gtf[df_gtf["attribute"].str.contains(tx_pattern, regex=True, na=False)].copy()

# Extract parent gene_id from transcript attributes
transcript_entries["gene_id"] = transcript_entries["attribute"].str.extract(r'gene_id "([^"]+)"')
gene_ids = transcript_entries["gene_id"].dropna().unique()
parent_gene = gene_ids[0] if len(gene_ids) else None
print(f"Parent gene_id: {parent_gene}" if parent_gene else "Parent gene_id: not found")

# Get gene entry
g_pattern = rf'gene_id "{re.escape(parent_gene)}"'
gene_entries = df_gtf[df_gtf["attribute"].str.contains(g_pattern, regex=True, na=False)].copy()

gene_entry_unique = gene_entries[gene_entries["feature"] == "gene"]

transcript_entries = pd.concat([gene_entry_unique, transcript_entries])
transcript_entries

Parent gene_id: LOC106360323


,seqname,source,feature,start,end,score,strand,frame,attribute,gene_id
919212,NC_027768.2,Gnomon,gene,8865505,8865905,.,+,.,"gene_id ""LOC106360323""; db_xref ""GeneID:106360...",NaN
919213,NC_027768.2,Gnomon,exon,8865505,8865905,.,+,.,"gene_id ""LOC106360323""; transcript_id ""XM_0137...",LOC106360323
919214,NC_027768.2,Gnomon,CDS,8865622,8865798,.,+,0,"gene_id ""LOC106360323""; transcript_id ""XM_0137...",LOC106360323
919215,NC_027768.2,Gnomon,start_codon,8865622,8865624,.,+,0,"gene_id ""LOC106360323""; transcript_id ""XM_0137...",LOC106360323
919216,NC_027768.2,Gnomon,stop_codon,8865799,8865801,.,+,0,"gene_id ""LOC106360323""; transcript_id ""XM_0137...",LOC106360323


In [ ]:
# Save this to a GTF file


### 3.3 Get assembly directly from trancript

In [ ]:
handle = Entrez.esummary(db="nuccore", id=ACCESSION, retmode="xml")
record = Entrez.read(handle)
handle.close()

assembly = record[0].get("AssemblyAcc", "N/A")
print(f"{ACCESSION} -> {assembly}")

XM_013799913.2 -> N/A


In [ ]:

# ── Diagnose batch vs single-ID esummary for actual withdrawn accessions ──────
# Test accessions from results/ncbi_genbank_unresolved.tsv
TEST_ACCS = [
    "XM_018653159.1",   # withdrawn (from unresolved.tsv)
    "XM_018654233.1",   # withdrawn
    "XM_018654288.1",   # withdrawn
]

print("── Single-ID call ──────────────────────────────────────────────")
h = Entrez.esummary(db="nuccore", id=TEST_ACCS[0], retmode="xml")
r = Entrez.read(h); h.close()
print(f"Type returned   : {type(r)}")
print(f"Keys for r[0]   : {list(r[0].keys())}")
print(f"AccessionVersion: {r[0].get('AccessionVersion')}")
print(f"Caption         : {r[0].get('Caption')}")
print(f"AssemblyAcc     : {r[0].get('AssemblyAcc')}")

print()
print("── Batch call (3 IDs comma-joined) ─────────────────────────────")
h = Entrez.esummary(db="nuccore", id=",".join(TEST_ACCS), retmode="xml")
rb = Entrez.read(h); h.close()
print(f"Type returned   : {type(rb)}")
print(f"Keys for rb[0]  : {list(rb[0].keys())}")
for rec in rb:
    print(f"  Caption={rec.get('Caption')!r:30}  AccessionVersion={rec.get('AccessionVersion')!r:20}  AssemblyAcc={rec.get('AssemblyAcc')!r}")


── Single-ID call ──────────────────────────────────────────────
Type returned   : <class 'Bio.Entrez.Parser.ListElement'>
Keys for r[0]   : ['Item', 'Id', 'Caption', 'Title', 'Extra', 'Gi', 'CreateDate', 'UpdateDate', 'Flags', 'TaxId', 'Length', 'Status', 'ReplacedBy', 'Comment', 'AccessionVersion']
AccessionVersion: XM_018653159.1
Caption         : XM_018653159
AssemblyAcc     : None

── Batch call (3 IDs comma-joined) ─────────────────────────────
Type returned   : <class 'Bio.Entrez.Parser.ListElement'>
Keys for rb[0]  : ['Item', 'Id', 'Caption', 'Title', 'Extra', 'Gi', 'CreateDate', 'UpdateDate', 'Flags', 'TaxId', 'Length', 'Status', 'ReplacedBy', 'Comment', 'AccessionVersion']
  Caption='XM_018653159'                  AccessionVersion='XM_018653159.1'      AssemblyAcc=None
  Caption='XM_018654233'                  AccessionVersion='XM_018654233.1'      AssemblyAcc=None
  Caption='XM_018654288'                  AccessionVersion='XM_018654288.1'      AssemblyAcc=None


In [ ]:

# ── Compare Status / ReplacedBy for both accession flavors ───────────────────
print("── Notebook 'Obsolete' accession (should have AssemblyAcc) ─────")
h = Entrez.esummary(db="nuccore", id="XM_013799913.2", retmode="xml")
r_obs = Entrez.read(h); h.close()
rec_obs = r_obs[0]
print(f"Keys         : {list(rec_obs.keys())}")
print(f"AssemblyAcc  : {rec_obs.get('AssemblyAcc')!r}")
print(f"Status       : {rec_obs.get('Status')!r}")
print(f"ReplacedBy   : {rec_obs.get('ReplacedBy')!r}")
print(f"Extra        : {str(rec_obs.get('Extra', ''))[:120]}")

print()
print("── Actual withdrawn accession (no AssemblyAcc) ─────────────────")
h2 = Entrez.esummary(db="nuccore", id="XM_018653159.1", retmode="xml")
r_w = Entrez.read(h2); h2.close()
rec_w = r_w[0]
print(f"Keys         : {list(rec_w.keys())}")
print(f"AssemblyAcc  : {rec_w.get('AssemblyAcc')!r}")
print(f"Status       : {rec_w.get('Status')!r}")
print(f"ReplacedBy   : {rec_w.get('ReplacedBy')!r}")
print(f"Extra        : {str(rec_w.get('Extra', ''))[:120]}")


── Notebook 'Obsolete' accession (should have AssemblyAcc) ─────


Keys         : ['Item', 'Id', 'Caption', 'Title', 'Extra', 'Gi', 'CreateDate', 'UpdateDate', 'Flags', 'TaxId', 'Length', 'Status', 'ReplacedBy', 'Comment', 'AccessionVersion']
AssemblyAcc  : None
Status       : 'suppressed'
ReplacedBy   : ''
Extra        : gi|1249722608|ref|XM_013799913.2|[1249722608]

── Actual withdrawn accession (no AssemblyAcc) ─────────────────
Keys         : ['Item', 'Id', 'Caption', 'Title', 'Extra', 'Gi', 'CreateDate', 'UpdateDate', 'Flags', 'TaxId', 'Length', 'Status', 'ReplacedBy', 'Comment', 'AccessionVersion']
AssemblyAcc  : None
Status       : 'replaced'
ReplacedBy   : 'XM_018653159.2'
Extra        : gi|1079425525|ref|XM_018653159.1|[1079425525]


In [ ]:

# ── Follow ReplacedBy chain → does current version have AssemblyAcc? ─────────
replacement = "XM_018653159.2"   # value of ReplacedBy for XM_018653159.1

h = Entrez.esummary(db="nuccore", id=replacement, retmode="xml")
r_rep = Entrez.read(h); h.close()
rec_rep = r_rep[0]
print(f"Accession    : {rec_rep.get('AccessionVersion')!r}")
print(f"Status       : {rec_rep.get('Status')!r}")
print(f"AssemblyAcc  : {rec_rep.get('AssemblyAcc')!r}")
print(f"Organism     : {rec_rep.get('Organism')!r}")
print(f"ReplacedBy   : {rec_rep.get('ReplacedBy')!r}")
print(f"Keys         : {list(rec_rep.keys())}")


Accession    : 'XM_018653159.2'
Status       : 'live'
AssemblyAcc  : None
Organism     : None
ReplacedBy   : ''
Keys         : ['Item', 'Id', 'Caption', 'Title', 'Extra', 'Gi', 'CreateDate', 'UpdateDate', 'Flags', 'TaxId', 'Length', 'Status', 'ReplacedBy', 'Comment', 'AccessionVersion']


In [ ]:

# ── Inspect nested Item elements and full record dump ─────────────────────────
print("Full record for XM_018653159.2:")
for key, val in rec_rep.items():
    if key == "Item":
        for item in val:
            if hasattr(item, 'attributes'):
                print(f"  Item[{item.attributes.get('Name','?')}] = {str(item)[:80]!r}")
    else:
        print(f"  {key} = {str(val)[:80]!r}")

print()
print("Same for the notebook's obsolete (suppressed) accession:")
for key, val in rec_obs.items():
    if key == "Item":
        for item in val:
            if hasattr(item, 'attributes'):
                print(f"  Item[{item.attributes.get('Name','?')}] = {str(item)[:80]!r}")
    else:
        print(f"  {key} = {str(val)[:80]!r}")


Full record for XM_018653159.2:
  Id = '1827912117'
  Caption = 'XM_018653159'
  Title = 'PREDICTED: Brassica rapa uncharacterized LOC108869124 (LOC108869124), transcript'
  Extra = 'gi|1827912117|ref|XM_018653159.2|[1827912117]'
  Gi = 'IntegerElement(1827912117, attributes={})'
  CreateDate = '2016/10/13'
  UpdateDate = '2020/12/07'
  Flags = 'IntegerElement(512, attributes={})'
  TaxId = 'IntegerElement(3711, attributes={})'
  Length = 'IntegerElement(1400, attributes={})'
  Status = 'live'
  ReplacedBy = ''
  Comment = '  '
  AccessionVersion = 'XM_018653159.2'

Same for the notebook's obsolete (suppressed) accession:
  Id = '1249722608'
  Caption = 'XM_013799913'
  Title = 'PREDICTED: Brassica napus arabinogalactan peptide 13 (LOC106360323), mRNA'
  Extra = 'gi|1249722608|ref|XM_013799913.2|[1249722608]'
  Gi = 'IntegerElement(1249722608, attributes={})'
  CreateDate = '2015/08/31'
  UpdateDate = '2017/10/04'
  Flags = 'IntegerElement(512, attributes={})'
  TaxId = 'IntegerElement

In [ ]:
if assembly == "N/A":
    handle = Entrez.elink(dbfrom="nuccore", db="assembly", id=ACCESSION)
    record = Entrez.read(handle)
    handle.close()

    links = record[0]["LinkSetDb"]
    if links:
        assembly_uid = links[0]["Link"][0]["Id"]
        print(f"Assembly UID: {assembly_uid}")

In [ ]:
record[0]

{'Item': [], 'Id': '1249722608', 'Caption': 'XM_013799913', 'Title': 'PREDICTED: Brassica napus arabinogalactan peptide 13 (LOC106360323), mRNA', 'Extra': 'gi|1249722608|ref|XM_013799913.2|[1249722608]', 'Gi': IntegerElement(1249722608, attributes={}), 'CreateDate': '2015/08/31', 'UpdateDate': '2017/10/04', 'Flags': IntegerElement(512, attributes={}), 'TaxId': IntegerElement(3708, attributes={}), 'Length': IntegerElement(401, attributes={}), 'Status': 'suppressed', 'ReplacedBy': '', 'Comment': ' This record was removed as a result of standard genome annotation processing. Please see www.ncbi.nlm.nih.gov/genome/annotation_euk/process/ for more information. ', 'AccessionVersion': 'XM_013799913.2'}

In [ ]:
accessions = [ACCESSION]
batch_size = min(50, len(accessions))

for start in range(0, len(accessions), batch_size):
    fetch_handle = Entrez.esummary(
        db="nuccore",
        query_key=query_key,
        WebEnv=webenv,
        retstart=start,
        retmax=batch_size,
        retmode="xml"
    )
    records = Entrez.read(fetch_handle)
    fetch_handle.close()

    for rec in records:
        mapping[rec["AccessionVersion"]] = rec.get("AssemblyAcc", "N/A")

## 4. Generate a GTF-Like Exon Table
GTF format is 1-based and closed. Biopython uses 0-based half-open intervals, so we add +1 to the start.

In [ ]:
rows = []
for feature in record.features:
    if feature.type == 'exon':
        start  = int(feature.location.start) + 1  # convert to 1-based
        end    = int(feature.location.end)
        strand = '+' if feature.location.strand == 1 else '-'
        gene   = feature.qualifiers.get('gene', ['?'])[0]
        number = feature.qualifiers.get('number', ['?'])[0]
        rows.append({
            'seqname'    : record.id,
            'source'     : 'GenBank',
            'feature'    : 'exon',
            'start'      : start,
            'end'        : end,
            'score'      : '.',
            'strand'     : strand,
            'frame'      : '.',
            'gene_id'    : gene,
            'transcript_id': record.id,
            'exon_number': number
        })

df_exons = pd.DataFrame(rows)
print(f'Found {len(df_exons)} exons')
df_exons.head()


Found 0 exons


""


In [ ]:
%%skip
# Save as a proper GTF file
gtf_cols = ['seqname','source','feature','start','end','score','strand','frame','gene_id','transcript_id','exon_number']

def format_attributes(row):
    return (f'gene_id "{row.gene_id}"; '
            f'transcript_id "{row.transcript_id}"; '
            f'exon_number "{row.exon_number}";')

with open('exons_from_genbank.gtf', 'w') as f:
    for _, row in df_exons.iterrows():
        attrs = format_attributes(row)
        f.write('\t'.join([
            row.seqname, row.source, row.feature,
            str(row.start), str(row.end),
            row.score, row.strand, row.frame, attrs
        ]) + '\n')
print('Saved: exons_from_genbank.gtf')


UsageError: Cell magic `%%skip` not found.


## 5. Find the Linked Gene ID (Entrez Gene)
The GenBank record's `db_xref` qualifiers often link directly to the NCBI Gene database.

In [ ]:
gene_ids = []
for feature in record.features:
    if feature.type in ('gene', 'mRNA'):
        for xref in feature.qualifiers.get('db_xref', []):
            if xref.startswith('GeneID:'):
                gene_ids.append(xref.split(':')[1])

gene_ids = list(set(gene_ids))
print('Linked Gene IDs:', gene_ids)


Linked Gene IDs: ['103278012']


## 6. Fetch the Gene Record
Use the Gene ID to retrieve full gene-level metadata from NCBI Gene.

In [ ]:
if gene_ids:
    handle = Entrez.efetch(db='gene', id=gene_ids[0], rettype='gene_table', retmode='text')
    print(handle.read()[:4000])  # print first 4000 chars
    handle.close()


atp5mj ATP synthase membrane subunit j[Anolis carolinensis]
Gene ID: 103278012, updated on 11-Mar-2024


Reference rAnoCar3.1.pri NC_085841.1  (minus strand) from: 351164075 to: 351157633
mRNA  XM_008105585.2, 4 exons,  total annotated spliced exon length: 478
protein  XP_008103792.1, 3 coding  exons,  annotated AA length: 56

Exon table for  mRNA  XM_008105585.2 and protein XP_008103792.1
Genomic Interval Exon		Genomic Interval Coding		Gene Interval Exon		Gene Interval Coding		Exon Length	Coding Length	Intron Length
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
351164075-351163922		1-154		154		2936
351160985-351160862		351160985-351160862		3091-3214		3091-3214		124		124		818
351160043-351160017		351160043-351160017		4033-4059		4033-4059		27		27		2211
351157805-351157633		351157805-351157786		6271-6443		6271-6290		173		20



## 7. Fetch the Protein Product (XP_ accession)
The CDS feature in an XM_ record links to the corresponding XP_ protein record.

In [ ]:
protein_id = None
for feature in record.features:
    if feature.type == 'CDS':
        protein_id = feature.qualifiers.get('protein_id', [None])[0]
        product    = feature.qualifiers.get('product', ['?'])[0]
        print(f'Protein ID : {protein_id}')
        print(f'Product    : {product}')
        break

if protein_id:
    handle = Entrez.efetch(db='protein', id=protein_id, rettype='gb', retmode='text')
    prot_record = SeqIO.read(handle, 'genbank')
    handle.close()
    print(f'\nProtein length : {len(prot_record.seq)} aa')
    print(f'Protein seq    : {prot_record.seq[:60]}...')


Protein ID : XP_008103792.1
Product    : 6.8 kDa mitochondrial proteolipid

Protein length : 56 aa
Protein seq    : MLRSAVQRWWDTHKIYHTQVFQELWVGIILTGYVYYKISYGGKKSVEGSKKSSGHH...


## 8. Search NCBI by Keyword
Use `Entrez.esearch` to find records matching a biological query, then `efetch` to retrieve them.

In [ ]:
# Search for mRNA records of a gene of interest
search_term = 'Homo sapiens[Organism] AND BRCA1[Gene] AND mRNA[Filter]'

handle  = Entrez.esearch(db='nucleotide', term=search_term, retmax=5)
results = Entrez.read(handle)
handle.close()

print(f'Total hits : {results["Count"]}')
print(f'Top 5 IDs  : {results["IdList"]}')


## 9. Fetch Multiple Records at Once
`efetch` accepts a comma-separated list of IDs for batch retrieval.

In [ ]:
ids = ','.join(results['IdList'])
handle  = Entrez.efetch(db='nucleotide', id=ids, rettype='gb', retmode='text')
records = list(SeqIO.parse(handle, 'genbank'))
handle.close()

for r in records:
    print(f'{r.id:<25} {len(r.seq):>8} bp  {r.description[:60]}')


## 10. Use elink to Cross-Reference Databases
`elink` lets you jump between NCBI databases — e.g., from a nucleotide record to its PubMed citations.

In [ ]:
# Find PubMed articles linked to our transcript
handle  = Entrez.elink(dbfrom='nucleotide', db='pubmed', id=GI_NUMBER)
link_results = Entrez.read(handle)
handle.close()

pmids = [link['Id'] for linkset in link_results
                     for links in linkset.get('LinkSetDb', [])
                     for link in links['Link']]
print(f'Linked PubMed IDs: {pmids[:10]}')
